In [ ]:
!pip install -q -U earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.5/481.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
google-genai 2.12.1 requires google-auth[requests]<2.56.0,>=2.48.1, but you have google-auth 2.56.3 which is incompatible.


In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
study_area = ee.Geometry.Rectangle([
    78.20, 17.25,
    78.65, 17.60
])

print("Study area created!")

Study area created!


In [ ]:
def mask_s2_clouds(image):
    """
    Masks clouds and cirrus in Sentinel-2 imagery
    using the QA60 band.
    """

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return image.updateMask(mask).divide(10000)


dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(study_area)
    .filterDate('2020-01-01', '2020-01-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
)

print("Number of Sentinel-2 images:", dataset.size().getInfo())

Number of Sentinel-2 images: 8


In [ ]:
composite = dataset.median().clip(study_area)

In [ ]:
def get_sentinel_composite(start_date, end_date, region):

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
    )

    print(
        start_date,
        "to",
        end_date,
        "| Images:",
        collection.size().getInfo()
    )

    composite = collection.median().clip(region)

    return composite

In [ ]:
image_2020 = get_sentinel_composite(
    '2020-01-01',
    '2020-01-31',
    study_area
)

2020-01-01 to 2020-01-31 | Images: 8


In [ ]:
visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2']
}

m = geemap.Map()

m.centerObject(study_area, 10)

m.addLayer(
    composite,
    visualization,
    'Sentinel-2 RGB - Hyderabad - 2020'
)

m.addLayer(
    study_area,
    {},
    'Hyderabad Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
image_2025 = get_sentinel_composite(
    '2025-01-01',
    '2025-01-31',
    study_area
)

2025-01-01 to 2025-01-31 | Images: 7


In [ ]:
# Visualization settings
visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2']
}

# Create map
m = geemap.Map()

# Center on Hyderabad
m.centerObject(study_area, 10)

# Add 2020
m.add_layer(
    image_2020,
    visualization,
    'Sentinel-2 RGB - 2020'
)

# Add 2025
m.add_layer(
    image_2025,
    visualization,
    'Sentinel-2 RGB - 2025'
)

# Add study area
m.add_layer(
    study_area,
    {},
    'Hyderabad Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
print("2020 bands:")
print(image_2020.bandNames().getInfo())

print("\n2025 bands:")
print(image_2025.bandNames().getInfo())

2020 bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']

2025 bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [ ]:
collection_2020 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(study_area)
    .filterDate('2020-01-01', '2020-01-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

collection_2025 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(study_area)
    .filterDate('2025-01-01', '2025-01-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

print("2020 images:", collection_2020.size().getInfo())
print("2025 images:", collection_2025.size().getInfo())

2020 images: 8
2025 images: 7


In [ ]:
print("2020 valid pixels:")
print(
    image_2020.select('B4')
    .reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=study_area,
        scale=100,
        maxPixels=1e9
    )
    .getInfo()
)

print("\n2025 valid pixels:")
print(
    image_2025.select('B4')
    .reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=study_area,
        scale=100,
        maxPixels=1e9
    )
    .getInfo()
)

2020 valid pixels:
{'B4': 194889}

2025 valid pixels:
{'B4': 194889}


In [ ]:
# ============================================
# CREATE SPECTRAL INDICES
# ============================================

def add_indices(image):
    """
    Adds NDVI, NDWI and NDBI to a Sentinel-2 image.

    NDVI = vegetation
    NDWI = water
    NDBI = built-up areas
    """

    # NDVI = (NIR - Red) / (NIR + Red)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    # NDWI = (Green - NIR) / (Green + NIR)
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    # NDBI = (SWIR - NIR) / (SWIR + NIR)
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')

    return image.addBands([ndvi, ndwi, ndbi])


# Add indices to both years
image_2020_features = add_indices(image_2020)
image_2025_features = add_indices(image_2025)

print("2020 features:")
print(image_2020_features.bandNames().getInfo())

print("\n2025 features:")
print(image_2025_features.bandNames().getInfo())

2020 features:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'NDVI', 'NDWI', 'NDBI']

2025 features:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'NDVI', 'NDWI', 'NDBI']


##NDVI

In [ ]:
ndvi_vis = {
    'min': -1,
    'max': 1,
    'palette': [
        'blue',
        'white',
        'green'
    ]
}

m_ndvi = geemap.Map()
m_ndvi.centerObject(study_area, 10)

m_ndvi.add_layer(
    image_2020_features.select('NDVI'),
    ndvi_vis,
    'NDVI 2020'
)

m_ndvi.add_layer(
    image_2025_features.select('NDVI'),
    ndvi_vis,
    'NDVI 2025'
)

m_ndvi

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

##NDWI

In [ ]:
ndwi_vis = {
    'min': -1,
    'max': 1,
    'palette': [
        'brown',
        'white',
        'blue'
    ]
}

m_ndwi = geemap.Map()
m_ndwi.centerObject(study_area, 10)

m_ndwi.add_layer(
    image_2020_features.select('NDWI'),
    ndwi_vis,
    'NDWI 2020'
)

m_ndwi.add_layer(
    image_2025_features.select('NDWI'),
    ndwi_vis,
    'NDWI 2025'
)

m_ndwi

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

##NDBI

In [ ]:
ndbi_vis = {
    'min': -1,
    'max': 1,
    'palette': [
        'green',
        'white',
        'red'
    ]
}

m_ndbi = geemap.Map()
m_ndbi.centerObject(study_area, 10)

m_ndbi.add_layer(
    image_2020_features.select('NDBI'),
    ndbi_vis,
    'NDBI 2020'
)

m_ndbi.add_layer(
    image_2025_features.select('NDBI'),
    ndbi_vis,
    'NDBI 2025'
)

m_ndbi

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…